In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:

import matplotlib.pyplot as plt

# delivery_time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery_time distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_copy = df.copy()
df_copy = df.drop(['Order_ID'], axis=1)
df_copy

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_copy)


In [ ]:
df_copy = df_copy.dropna(subset=['Delivery_Time'])
check_missing_values(df_copy)

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_copy)

In [ ]:
check_missing_values(df_copy)

In [ ]:
# Task 4: Write your code here:
# 3. Do we have categorical columns?
categorical_cols = df_copy.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_copy[col] = le.fit_transform(df_copy[col])
  label_encoders[col] = le

df_copy

In [ ]:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

onehot_encoders = {}
for col in categorical_cols:
  ol = OneHotEncoder()  # Instantiate OneHotEncoder
  df_copy[col] = ol.fit_transform(df_copy[col])
  onehot_encoders[col] = ol

df_copy

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_copy.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_copy[numerical_cols] = scaler.fit_transform(df_copy[numerical_cols])
df_copy.head()

In [ ]:
# Task 6: Write your code here:
# 1. Is the target imbalanced?
# we do not need this bec our data is numorical

In [ ]:
# Task 1: Write your code here:
X = df_copy.drop("Delivery_Time", axis=1).astype(float)
y = df_copy['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

lr_losses = []

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

model = RandomForestRegressor(n_estimators=200)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
  model.fit(X_train, y_train)

    # Predic
  y_pred = model.predict(X_test)

    # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)

    # Store results
  lr_losses.append(mae)

# Calculate average loss across folds
print(f"\nMAE:  {np.mean(lr_losses, axis=0):.4f}")

In [ ]:
# Task 1: Write your code here:
importances = []
importances = model.feature_importances_

plt.figure(figsize=(10, 5))
features = X.columns
plt.barh(features, importances)
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Score')
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, edgecolor='black')
plt.title('Random Forest: predicted delivery time')
plt.xlabel('Predicted y_pred (delivery_time)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: